In [ ]:
import csv
import re

# ── Input file paths ───────────────────────────────────────────────────────────
INCOME_CSV  = "ResidentWorkingPersonsAged15YearsandOverbyPlanningAreaandGrossMonthlyIncomefromWorkGeneralHouseholdSurvey2015.csv"
ANNUAL_CSV  = "SingaporeResidentsByPlanningRegionAgeGroupAndSexEndJuneAnnual.csv"
CENSUS_CSV  = "ResidentPopulationbyPlanningAreaSubzoneofResidenceAgeGroupandSexCensusofPopulation2020.csv"
OUTPUT_CSV  = "singapore_planning_area_demographics.csv"

# ── Planning Area → Region lookup ──────────────────────────────────────────────
PA_TO_REGION = {
    "ANG MO KIO": "CENTRAL", "BISHAN": "CENTRAL", "BUKIT MERAH": "CENTRAL",
    "BUKIT TIMAH": "CENTRAL", "DOWNTOWN CORE": "CENTRAL", "GEYLANG": "CENTRAL",
    "KALLANG": "CENTRAL", "MARINE PARADE": "CENTRAL", "MARINA EAST": "CENTRAL",
    "MARINA SOUTH": "CENTRAL", "MUSEUM": "CENTRAL", "NEWTON": "CENTRAL",
    "NOVENA": "CENTRAL", "ORCHARD": "CENTRAL", "OUTRAM": "CENTRAL",
    "QUEENSTOWN": "CENTRAL", "RIVER VALLEY": "CENTRAL", "ROCHOR": "CENTRAL",
    "SINGAPORE RIVER": "CENTRAL", "SOUTHERN ISLANDS": "CENTRAL",
    "STRAITS VIEW": "CENTRAL", "TANGLIN": "CENTRAL", "TOA PAYOH": "CENTRAL",
    "BEDOK": "EAST", "CHANGI": "EAST", "CHANGI BAY": "EAST",
    "PASIR RIS": "EAST", "PAYA LEBAR": "EAST", "TAMPINES": "EAST",
    "CENTRAL WATER CATCHMENT": "NORTH", "LIM CHU KANG": "NORTH",
    "MANDAI": "NORTH", "SEMBAWANG": "NORTH", "SIMPANG": "NORTH",
    "SUNGEI KADUT": "NORTH", "WOODLANDS": "NORTH", "YISHUN": "NORTH",
    "HOUGANG": "NORTH-EAST", "NORTH-EASTERN ISLANDS": "NORTH-EAST",
    "PUNGGOL": "NORTH-EAST", "SELETAR": "NORTH-EAST",
    "SENGKANG": "NORTH-EAST", "SERANGOON": "NORTH-EAST",
    "BOON LAY": "WEST", "BUKIT BATOK": "WEST", "BUKIT PANJANG": "WEST",
    "CHOA CHU KANG": "WEST", "CLEMENTI": "WEST", "JURONG EAST": "WEST",
    "JURONG WEST": "WEST", "PIONEER": "WEST", "TENGAH": "WEST",
    "TUAS": "WEST", "WESTERN ISLANDS": "WEST", "WESTERN WATER CATCHMENT": "WEST",
}


def safe_int(v):
    try:
        return int(v)
    except (ValueError, TypeError):
        return 0


def safe_float(v):
    try:
        return float(v)
    except (ValueError, TypeError):
        return 0.0


# ── 1. Income (GHS 2015, Planning Area level) ──────────────────────────────────
# Rows are one per Planning Area. Values are worker counts in thousands.
# Income brackets are consolidated into broader bands for usability.
income = {}
with open(INCOME_CSV) as f:
    for row in csv.DictReader(f):
        pa = row["Thousands"].strip().upper()
        if pa in ("TOTAL", "OTHERS"):
            continue
        income[pa] = {
            "income_survey_year":              "2015",
            "income_total_workers_thousands":  row["Total"],
            "income_below_1000":               row["Below_1_000"],
            "income_1000_1999":                str(round(safe_float(row["1_000_1_499"]) + safe_float(row["1_500_1_999"]), 1)),
            "income_2000_2999":                str(round(safe_float(row["2_000_2_499"]) + safe_float(row["2_500_2_999"]), 1)),
            "income_3000_3999":                row["3_000_3_999"],
            "income_4000_4999":                row["4_000_4_999"],
            "income_5000_9999":                str(round(safe_float(row["5_000_5_999"]) + safe_float(row["6_000_6_999"]) + safe_float(row["7_000_7_999"]) + safe_float(row["8_000_8_999"]) + safe_float(row["9_000_9_999"]), 1)),
            "income_10000_over":               str(round(safe_float(row["10_000_10_999"]) + safe_float(row["11_000_11_999"]) + safe_float(row["12_000andOver"]), 1)),
        }


# ── 2. Census 2020 (Planning Area + Subzone) ───────────────────────────────────
# PA-level rows are identified by the pattern "Name - Total".
# Subzone rows (no "- Total" suffix) are skipped.
# Age groups are consolidated into three broad bands: 0–14, 15–64, 65+.
census = {}
with open(CENSUS_CSV) as f:
    for row in csv.DictReader(f):
        m = re.match(r"^(.+?)\s*-\s*Total$", row["Number"].strip())
        if not m:
            continue
        pa = m.group(1).strip().upper()
        census[pa] = {
            "pop2020_total":   row["Total_Total"],
            "pop2020_male":    row["Males_Total"],
            "pop2020_female":  row["Females_Total"],
            "pop2020_0_14":    str(safe_int(row["Total_0_4"]) + safe_int(row["Total_5_9"]) + safe_int(row["Total_10_14"])),
            "pop2020_15_64":   str(sum(safe_int(row[f"Total_{g}"]) for g in ["15_19", "20_24", "25_29", "30_34", "35_39", "40_44", "45_49", "50_54", "55_59", "60_64"])),
            "pop2020_65plus":  str(sum(safe_int(row[f"Total_{g}"]) for g in ["65_69", "70_74", "75_79", "80_84", "85_89", "90andOver"])),
        }


# ── 3. Annual regional totals (2019–2025) ──────────────────────────────────────
# This file is structured hierarchically: a Region header row, followed by
# indented age-group rows. Only the Region header rows (no leading space,
# no "(Male)"/"(Female)" suffix) are extracted.
# Region names are normalised by stripping " Region" so they match PA_TO_REGION values.
annual = {}
with open(ANNUAL_CSV) as f:
    for row in csv.DictReader(f):
        ds = row["DataSeries"].strip()
        if ds.startswith(" ") or "(" in ds:
            continue
        region_key = ds.upper().replace(" REGION", "").strip()
        annual[region_key] = {
            "region_total_pop_2019": row.get("2019", ""),
            "region_total_pop_2020": row.get("2020", ""),
            "region_total_pop_2021": row.get("2021", ""),
            "region_total_pop_2022": row.get("2022", ""),
            "region_total_pop_2023": row.get("2023", ""),
            "region_total_pop_2024": row.get("2024", ""),
            "region_total_pop_2025": row.get("2025", ""),
        }


# ── 4. Merge ───────────────────────────────────────────────────────────────────
# Census is used as the master list of Planning Areas (most complete).
# Income and regional totals are left-joined in; unmatched PAs get blank values.
OUTPUT_FIELDS = [
    "planning_area", "region",
    "pop2020_total", "pop2020_male", "pop2020_female",
    "pop2020_0_14", "pop2020_15_64", "pop2020_65plus",
    "income_survey_year", "income_total_workers_thousands",
    "income_below_1000", "income_1000_1999", "income_2000_2999",
    "income_3000_3999", "income_4000_4999", "income_5000_9999", "income_10000_over",
    "region_total_pop_2019", "region_total_pop_2020", "region_total_pop_2021",
    "region_total_pop_2022", "region_total_pop_2023", "region_total_pop_2024",
    "region_total_pop_2025",
]

rows = []
for pa in sorted(census.keys()):
    region = PA_TO_REGION.get(pa, "UNKNOWN")
    row = {"planning_area": pa, "region": region}
    row.update(census.get(pa, {}))
    row.update(income.get(pa, {}))
    row.update(annual.get(region, {}))
    rows.append(row)

with open(OUTPUT_CSV, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=OUTPUT_FIELDS, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(rows)

# ── Summary ────────────────────────────────────────────────────────────────────
income_matched = sum(1 for r in rows if r.get("income_total_workers_thousands"))
print(f"Output: {OUTPUT_CSV}")
print(f"  Planning areas : {len(rows)}")
print(f"  With census    : {len(rows)}  (all)")
print(f"  With income    : {income_matched}  (residential PAs only — others expected blank)")